In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2017_2018"
    / "Questionnnair"
)

xpt_file = DATA_DIR / "MCQ_J.xpt"
csv_file = DATA_DIR / "MCQ_J_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve completely untouched pandas-decoded source.
#
# IMPORTANT:
#
# MCQ_J requires NO validated numeric correction.
#
# Therefore source_df becomes the exact numeric baseline
# against which the cleaned dataframe and final CSV
# will be compared.
#

source_df = df.copy()


print("=" * 90)
print("NHANES 2017-2018 MCQ_J XPT -> CSV")
print("=" * 90)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "MCQ010",
    "MCQ025",
    "MCQ035",
    "MCQ040",
    "MCQ050",
    "AGQ030",
    "MCQ053",
    "MCQ080",
    "MCQ092",
    "MCD093",
    "MCQ149",
    "MCQ151",
    "RHD018",
    "MCQ160A",
    "MCD180A",
    "MCQ195",
    "MCQ160N",
    "MCD180N",
    "MCQ160B",
    "MCD180B",
    "MCQ160C",
    "MCD180C",
    "MCQ160D",
    "MCD180D",
    "MCQ160E",
    "MCD180E",
    "MCQ160F",
    "MCD180F",
    "MCQ160M",
    "MCQ170M",
    "MCD180M",
    "MCQ160G",
    "MCD180G",
    "MCQ160K",
    "MCQ170K",
    "MCD180K",
    "MCQ160O",
    "MCQ160L",
    "MCQ170L",
    "MCD180L",
    "MCQ500",
    "MCQ510A",
    "MCQ510B",
    "MCQ510C",
    "MCQ510D",
    "MCQ510E",
    "MCQ510F",
    "MCQ520",
    "MCQ530",
    "MCQ540",
    "MCQ550",
    "MCQ560",
    "MCQ570",
    "MCQ203",
    "MCQ206",
    "MCQ220",
    "MCQ230A",
    "MCD240A",
    "MCQ230B",
    "MCD240B",
    "MCQ230C",
    "MCD240C",
    "MCQ230D",
    "MCQ300B",
    "MCQ300C",
    "MCQ300A",
    "MCQ366A",
    "MCQ366B",
    "MCQ366C",
    "MCQ366D",
    "MCQ371A",
    "MCQ371B",
    "MCQ371C",
    "MCQ371D",
    "OSQ230"
]


if df.columns.tolist() != EXPECTED_COLUMNS:

    raise ValueError(
        "MCQ_J column names/order differ from expected.\n\n"
        f"Expected:\n{EXPECTED_COLUMNS}\n\n"
        f"Actual:\n{df.columns.tolist()}"
    )


if len(df) != 8897:

    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 76:

    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 8,897 rows x 76 columns."
)


# ============================================================
# 4. FIELD TYPE PLAN
# ============================================================

#
# Attached MCQ_J was inspected BEFORE any conversion.
#
# No genuine continuous decimal variables were found.
#
# Variables consist of:
#
#     SEQN
#     questionnaire codes
#     ages
#     derived whole-number age/month variables
#     categorical response codes
#
# Every populated MCQ_J numeric value is whole-number.
#
# IMPORTANT:
#
# This conclusion applies ONLY to MCQ_J.
#
# It does NOT mean decimals should be removed from other
# NHANES files.
#
# Genuine Dietary, Laboratory, Examination, sample-weight,
# anthropometric, nutrient, exposure, etc. decimal variables
# must remain untouched.
#

DECIMAL_COLUMNS = []

INTEGER_COLUMNS = EXPECTED_COLUMNS.copy()


print("\n--- FIELD TYPE PLAN ---")

print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)

print(
    "Provisional whole-number/code fields:",
    len(INTEGER_COLUMNS)
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas/SAS XPORT decoding artifact:
#
#     5.397605346934028e-79
#
# Some NHANES XPT files contain this value where the
# intended value is zero.
#
# It must NEVER be replaced globally without first
# validating the variable and count against CDC.
#
# Direct inspection of attached MCQ_J:
#
#     Tiny values = 0
#
# Therefore NO tiny-value repair should occur.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


if numeric_cols != EXPECTED_COLUMNS:

    raise ValueError(
        "Expected all 76 MCQ_J fields "
        "to load numerically."
    )


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print("\n--- ORIGINAL XPT TINY-VALUE CHECK ---")

print(
    "Total tiny XPORT artifacts:",
    f"{tiny_total:,}"
)


if tiny_total > 0:

    print(
        tiny_counts[
            tiny_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


if tiny_total != 0:

    raise ValueError(
        "STOP: Unexpected tiny XPORT artifacts "
        "found in MCQ_J.\n\n"
        + tiny_counts[
            tiny_counts > 0
        ].to_string()
    )


print(
    "PASS: MCQ_J contains no tiny XPORT artifacts."
)


# ============================================================
# 6. CHECK ORIGINAL ORDINARY NUMERIC ZEROS
# ============================================================

#
# Direct inspection of attached MCQ_J also found:
#
#     ordinary numeric zeros = 0
#
# Therefore no zero values need restoration.
#

true_zero_counts = (
    df[numeric_cols]
    .eq(0)
    .sum()
)


true_zero_total = int(
    true_zero_counts.sum()
)


print("\n--- ORIGINAL NUMERIC ZERO CHECK ---")

print(
    "Ordinary numeric zeros:",
    f"{true_zero_total:,}"
)


if true_zero_total > 0:

    print(
        true_zero_counts[
            true_zero_counts > 0
        ]
        .sort_values(
            ascending=False
        )
        .to_string()
    )


if true_zero_total != 0:

    raise ValueError(
        "Unexpected ordinary numeric zero values "
        "found in MCQ_J."
    )


print(
    "PASS: Original MCQ_J contains no numeric zeros."
)


# ============================================================
# 7. EXPECTED CORRECTED SOURCE
# ============================================================

#
# MCQ_J requires NO numeric correction.
#
# Therefore:
#
#     expected_df == source_df
#
# numerically and in missing-value positions.
#

expected_df = source_df.copy()


# ============================================================
# 8. CHECK EVERY COLUMN FOR FRACTIONAL VALUES
# ============================================================

#
# This occurs BEFORE converting anything to Int64.
#
# np.round() below is used ONLY for testing whether
# values are integral.
#
# It NEVER changes df.
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) == 0:

        print(
            f"{col}: 0 populated observations"
        )

        continue

    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )

    fractional_count = int(
        fractional_mask.sum()
    )

    print(
        f"{col}: "
        f"{fractional_count:,} fractional observations"
    )

    if fractional_count > 0:

        fractional_columns.append(
            col
        )


if fractional_columns:

    raise ValueError(
        "STOP: Genuine fractional variables found.\n"
        "These variables will NOT be rounded or converted "
        "to Int64:\n\n"
        + str(fractional_columns)
    )


print(
    "\nPASS: No genuine fractional values in "
    "any of the 76 MCQ_J columns."
)


# ============================================================
# 9. VALIDATE SEQN
# ============================================================

seqn_values = (
    df["SEQN"]
    .dropna()
    .to_numpy(
        dtype=float
    )
)


if not np.isclose(
    seqn_values,
    np.round(seqn_values),
    rtol=0,
    atol=1e-12
).all():

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


print("\n--- SEQN ---")

print(
    "Minimum:",
    df["SEQN"].min()
)

print(
    "Maximum:",
    df["SEQN"].max()
)

print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)

print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if df["SEQN"].min() != 93703:

    raise ValueError(
        "Unexpected minimum SEQN."
    )


if df["SEQN"].max() != 102956:

    raise ValueError(
        "Unexpected maximum SEQN."
    )


if int(
    df["SEQN"]
    .isna()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains missing values."
    )


if int(
    df["SEQN"]
    .duplicated()
    .sum()
) != 0:

    raise ValueError(
        "SEQN contains duplicate values."
    )


print(
    "PASS: SEQN 93703-102956 validated."
)


# ============================================================
# 10. CONVERT ALL PROVEN WHOLE-NUMBER FIELDS TO Int64
# ============================================================

#
# Every variable was individually tested above.
#
# No variable contains genuine fractional values.
#
# Therefore Int64 is safe for MCQ_J.
#
# Examples:
#
#     93703.0 -> 93703
#     1.0     -> 1
#     99999.0 -> 99999
#
# Numeric meaning does not change.
#
# NO .round() is used during conversion.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )

    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )

        if not whole_mask.all():

            raise ValueError(
                f"{col} unexpectedly contains "
                "fractional values."
            )

    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: All 76 MCQ_J variables converted "
    "to nullable Int64 without rounding."
)


# ============================================================
# 11. CDC EXACT FREQUENCY CHECK HELPER
# ============================================================

def check_frequency(
    column,
    expected_values,
    expected_missing
):

    actual_values = (
        df[column]
        .value_counts()
        .sort_index()
        .to_dict()
    )

    actual_missing = int(
        df[column]
        .isna()
        .sum()
    )

    print(
        f"\n--- CDC CHECK: {column} ---"
    )

    print(
        "Observed:",
        actual_values
    )

    print(
        "Missing:",
        f"{actual_missing:,}"
    )

    if actual_values != expected_values:

        raise ValueError(
            f"{column} frequencies differ from CDC.\n\n"
            f"Expected:\n{expected_values}\n\n"
            f"Actual:\n{actual_values}"
        )

    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs from CDC.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )

    print(
        f"PASS: {column} matches CDC."
    )


# ============================================================
# 12. CDC EXACT FREQUENCY CHECKS
# ============================================================

#
# These provide independent CDC/codebook checks across
# the major sections of MCQ_J.
#
# Full XPT -> CSV fidelity for ALL 76 variables is
# performed later in the program.
#

CDC_EXACT_CHECKS = {

    "MCQ010": (
        {
            1: 1325,
            2: 7563,
            9: 9
        },
        0
    ),

    "MCQ035": (
        {
            1: 825,
            2: 471,
            7: 1,
            9: 28
        },
        7572
    ),

    "MCQ040": (
        {
            1: 359,
            2: 463,
            7: 1,
            9: 2
        },
        8072
    ),

    "MCQ050": (
        {
            1: 167,
            2: 658
        },
        8072
    ),

    "AGQ030": (
        {
            1: 196,
            2: 624,
            9: 5
        },
        8072
    ),

    "MCQ053": (
        {
            1: 339,
            2: 8543,
            7: 1,
            9: 14
        },
        0
    ),

    "MCQ080": (
        {
            1: 2304,
            2: 3850,
            9: 7
        },
        2736
    ),

    "MCQ092": (
        {
            1: 689,
            2: 7107,
            7: 1,
            9: 81
        },
        1019
    ),

    "MCD093": (
        {
            1: 95,
            2: 162,
            3: 411,
            9: 21
        },
        8208
    ),

    "MCQ160A": (
        {
            1: 1695,
            2: 3857,
            9: 17
        },
        3328
    ),

    "MCQ195": (
        {
            1: 711,
            2: 337,
            3: 26,
            4: 175,
            7: 3,
            9: 443
        },
        7202
    ),

    "MCQ160B": (
        {
            1: 201,
            2: 5351,
            9: 17
        },
        3328
    ),

    "MCQ160C": (
        {
            1: 265,
            2: 5288,
            9: 16
        },
        3328
    ),

    "MCQ160D": (
        {
            1: 161,
            2: 5378,
            9: 30
        },
        3328
    ),

    "MCQ160E": (
        {
            1: 270,
            2: 5289,
            9: 10
        },
        3328
    ),

    "MCQ160F": (
        {
            1: 273,
            2: 5286,
            9: 10
        },
        3328
    ),

    "MCQ520": (
        {
            1: 1201,
            2: 4363,
            9: 5
        },
        3328
    ),

    "MCQ530": (
        {
            1: 281,
            2: 647,
            3: 269,
            9: 4
        },
        7696
    ),

    "MCQ540": (
        {
            1: 808,
            2: 393
        },
        7696
    ),

    "MCQ550": (
        {
            1: 623,
            2: 4931,
            9: 15
        },
        3328
    ),

    "MCQ560": (
        {
            1: 641,
            2: 4925,
            9: 3
        },
        3328
    ),

    "MCQ203": (
        {
            1: 171,
            2: 7692,
            9: 15
        },
        1019
    ),

    "MCQ220": (
        {
            1: 588,
            2: 4979,
            9: 2
        },
        3328
    ),

    "MCQ300B": (
        {
            1: 2013,
            2: 5717,
            9: 148
        },
        1019
    ),

    "MCQ300C": (
        {
            1: 2624,
            2: 2863,
            7: 1,
            9: 81
        },
        3328
    ),

    "MCQ300A": (
        {
            1: 709,
            2: 4732,
            7: 1,
            9: 127
        },
        3328
    ),

    "MCQ366A": (
        {
            1: 1668,
            2: 4491,
            9: 2
        },
        2736
    ),

    "MCQ366B": (
        {
            1: 2474,
            2: 3686,
            9: 1
        },
        2736
    ),

    "MCQ366C": (
        {
            1: 1789,
            2: 4368,
            9: 4
        },
        2736
    ),

    "MCQ366D": (
        {
            1: 1914,
            2: 4243,
            9: 4
        },
        2736
    ),

    "MCQ371A": (
        {
            1: 3835,
            2: 2321,
            9: 5
        },
        2736
    ),

    "MCQ371B": (
        {
            1: 3622,
            2: 2536,
            9: 3
        },
        2736
    ),

    "MCQ371C": (
        {
            1: 3378,
            2: 2779,
            9: 4
        },
        2736
    ),

    "MCQ371D": (
        {
            1: 3572,
            2: 2583,
            9: 6
        },
        2736
    ),

    "OSQ230": (
        {
            1: 1128,
            2: 2732,
            9: 22
        },
        5015
    )
}


for column, (
    expected_values,
    expected_missing
) in CDC_EXACT_CHECKS.items():

    check_frequency(
        column,
        expected_values,
        expected_missing
    )


print(
    "\nPASS: All selected CDC exact-frequency "
    "checks passed."
)


# ============================================================
# 13. CDC RANGE-SUMMARY CHECK HELPER
# ============================================================

#
# Some MCQ variables are ages rather than simple
# categorical codes.
#
# CDC publishes ranges plus special codes instead
# of each individual age frequency.
#

def check_range_summary(
    column,
    lower,
    upper,
    expected_range_count,
    special_counts,
    expected_missing
):

    values = df[column]

    range_count = int(
        values.between(
            lower,
            upper,
            inclusive="both"
        )
        .sum()
    )

    actual_missing = int(
        values.isna()
        .sum()
    )

    print(
        f"\n--- CDC RANGE CHECK: {column} ---"
    )

    print(
        f"Range {lower}-{upper}:",
        range_count
    )

    if range_count != expected_range_count:

        raise ValueError(
            f"{column} range count differs from CDC.\n"
            f"Expected: {expected_range_count:,}\n"
            f"Actual:   {range_count:,}"
        )

    for code, expected_count in special_counts.items():

        actual_count = int(
            values.eq(code)
            .sum()
        )

        print(
            f"Code {code}:",
            actual_count
        )

        if actual_count != expected_count:

            raise ValueError(
                f"{column} code {code} count differs.\n"
                f"Expected: {expected_count:,}\n"
                f"Actual:   {actual_count:,}"
            )

    if actual_missing != expected_missing:

        raise ValueError(
            f"{column} missing count differs.\n"
            f"Expected: {expected_missing:,}\n"
            f"Actual:   {actual_missing:,}"
        )

    print(
        f"PASS: {column} CDC range summary matches."
    )


# ============================================================
# 14. CDC RANGE CHECK - MCQ025
# ============================================================

#
# Age when first had asthma.
#
# CDC:
#
#     1-79   = 1,296
#     80     = 8
#     99999  = 21
#     Missing = 7,572
#

check_range_summary(
    "MCQ025",
    1,
    79,
    1296,
    {
        80: 8,
        99999: 21
    },
    7572
)


# ============================================================
# 15. CDC RANGE CHECK - RHD018
# ============================================================

#
# Estimated age in months at menarche.
#
# CDC:
#
#     86-142 = 46
#     9999   = 9
#     Missing = 8,842
#

check_range_summary(
    "RHD018",
    86,
    142,
    46,
    {
        9999: 9
    },
    8842
)


# ============================================================
# 16. CDC RANGE CHECK - MCQ570
# ============================================================

#
# Age at first gallbladder surgery.
#
# CDC:
#
#     15-79 = 620
#     80 = 10
#     99999 = 11
#     Missing = 8,256
#

check_range_summary(
    "MCQ570",
    15,
    79,
    620,
    {
        80: 10,
        99999: 11
    },
    8256
)


# ============================================================
# 17. CDC RANGE CHECK - MCQ206
# ============================================================

#
# Age when told had jaundice.
#
# CDC:
#
#     1-76 = 171
#     Missing = 8,726
#

check_range_summary(
    "MCQ206",
    1,
    76,
    171,
    {},
    8726
)


# ============================================================
# 18. ROUTING CHECK HELPER
# ============================================================

#
# Validates deterministic questionnaire skip/routing
# relationships.
#

def check_route(
    parent_column,
    parent_code,
    child_column
):

    expected_present = (
        df[parent_column]
        .eq(parent_code)
        .fillna(False)
        .to_numpy(
            dtype=bool
        )
    )

    actual_present = (
        df[child_column]
        .notna()
        .to_numpy(
            dtype=bool
        )
    )

    mismatch_count = int(
        np.sum(
            expected_present
            !=
            actual_present
        )
    )

    expected_count = int(
        expected_present.sum()
    )

    actual_count = int(
        actual_present.sum()
    )

    print(
        f"\n--- ROUTING: "
        f"{parent_column}={parent_code} -> "
        f"{child_column} ---"
    )

    print(
        "Expected populated:",
        expected_count
    )

    print(
        "Actually populated:",
        actual_count
    )

    print(
        "Routing mismatches:",
        mismatch_count
    )

    if mismatch_count != 0:

        raise ValueError(
            f"Routing mismatch: "
            f"{parent_column}={parent_code} -> "
            f"{child_column}"
        )

    print(
        "PASS: Routing validated."
    )


# ============================================================
# 19. ASTHMA ROUTING CHECKS
# ============================================================

check_route(
    "MCQ010",
    1,
    "MCQ035"
)

check_route(
    "MCQ035",
    1,
    "MCQ040"
)

check_route(
    "MCQ035",
    1,
    "MCQ050"
)

check_route(
    "MCQ035",
    1,
    "AGQ030"
)


# ============================================================
# 20. BLOOD TRANSFUSION ROUTING
# ============================================================

check_route(
    "MCQ092",
    1,
    "MCD093"
)


# ============================================================
# 21. ARTHRITIS / MEDICAL CONDITION ROUTING
# ============================================================

ROUTING_PAIRS = [

    (
        "MCQ160A",
        "MCD180A"
    ),

    (
        "MCQ160A",
        "MCQ195"
    ),

    (
        "MCQ160N",
        "MCD180N"
    ),

    (
        "MCQ160B",
        "MCD180B"
    ),

    (
        "MCQ160C",
        "MCD180C"
    ),

    (
        "MCQ160D",
        "MCD180D"
    ),

    (
        "MCQ160E",
        "MCD180E"
    ),

    (
        "MCQ160F",
        "MCD180F"
    ),

    (
        "MCQ160M",
        "MCQ170M"
    ),

    (
        "MCQ160M",
        "MCD180M"
    ),

    (
        "MCQ160G",
        "MCD180G"
    ),

    (
        "MCQ160K",
        "MCQ170K"
    ),

    (
        "MCQ160K",
        "MCD180K"
    ),

    (
        "MCQ160L",
        "MCQ170L"
    ),

    (
        "MCQ160L",
        "MCD180L"
    )
]


for parent, child in ROUTING_PAIRS:

    check_route(
        parent,
        1,
        child
    )


# ============================================================
# 22. ABDOMINAL / GALLBLADDER ROUTING
# ============================================================

check_route(
    "MCQ520",
    1,
    "MCQ530"
)

check_route(
    "MCQ520",
    1,
    "MCQ540"
)

check_route(
    "MCQ560",
    1,
    "MCQ570"
)


# ============================================================
# 23. JAUNDICE / CANCER ROUTING
# ============================================================

check_route(
    "MCQ203",
    1,
    "MCQ206"
)

check_route(
    "MCQ220",
    1,
    "MCQ230A"
)


print(
    "\nPASS: Deterministic MCQ_J routing "
    "relationships validated."
)


# ============================================================
# 24. FINAL ZERO PATTERN BEFORE EXPORT
# ============================================================

#
# Source contained no numeric zero values.
#
# Since no numeric correction was performed,
# final zero pattern must also be empty.
#

final_zero_counts = {}


for col in EXPECTED_COLUMNS:

    count = int(
        df[col]
        .eq(0)
        .sum()
    )

    if count > 0:

        final_zero_counts[col] = count


print("\n--- FINAL ZERO PATTERN ---")

print(
    final_zero_counts
)


if final_zero_counts != {}:

    raise ValueError(
        "Unexpected zero values introduced:\n"
        + str(final_zero_counts)
    )


print(
    "PASS: No numeric zero values introduced."
)


# ============================================================
# 25. COMPLETE VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Critical fidelity test.
#
# MCQ_J permitted numeric modifications:
#
#     NONE
#
# Therefore every one of the 76 columns must still
# exactly match the original XPT-derived numeric values.
#

print(
    "\n--- COMPLETE PRE-EXPORT VALUE FIDELITY ---"
)


fidelity_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )

    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )

    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:

        fidelity_failures.append(
            col
        )


if fidelity_failures:

    raise ValueError(
        "Unexpected numeric changes before export:\n"
        + str(fidelity_failures)
    )


print(
    "PASS: All 76 MCQ_J variables exactly match "
    "their original XPT numeric values."
)


# ============================================================
# 26. MISSING-VALUE FIDELITY BEFORE EXPORT
# ============================================================

missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )

    current_missing = (
        df[col]
        .isna()
        .to_numpy()
    )

    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            current_missing
        )
    )


print(
    "\nMissing-position differences before export:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed "
        "before CSV export."
    )


print(
    "PASS: Missing positions unchanged."
)


# ============================================================
# 27. EXPORT CSV
# ============================================================

#
# Fidelity-preserving export.
#
# ABSOLUTELY NO:
#
#     df.round(...)
#
# ABSOLUTELY NO:
#
#     float_format=
#
# ABSOLUTELY NO blanket decimal formatting.
#
# MCQ_J itself has no genuine decimals, but retaining
# this rule protects the same pipeline when processing:
#
#     Dietary
#     Laboratory
#     Examination
#     exposure
#     weight
#     continuous numeric files
#
# where decimals must remain as decoded.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 28. RE-READ CSV WITH ROUND-TRIP FLOAT PARSER
# ============================================================

roundtrip_df = pd.read_csv(
    csv_file,
    low_memory=False,
    float_precision="round_trip"
)


if roundtrip_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed."
    )


if (
    roundtrip_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV columns/order changed."
    )


print(
    "PASS: CSV structure preserved."
)


# ============================================================
# 29. EXACT ORIGINAL XPT -> CSV NUMERIC FIDELITY
# ============================================================

#
# No MCQ_J numeric values were supposed to change.
#
# Therefore all 76 exported columns must round-trip
# exactly to original XPT numeric values.
#

print(
    "\n--- EXACT ORIGINAL XPT -> CSV FIDELITY ---"
)


csv_failures = []


for col in EXPECTED_COLUMNS:

    expected_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )

    exported_values = (
        roundtrip_df[col]
        .to_numpy(
            dtype=float
        )
    )

    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )

    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "XPT -> CSV numeric fidelity differences:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 76 CSV variables round-trip "
    "to exact original XPT numeric values."
)


# ============================================================
# 30. CSV MISSING-VALUE FIDELITY
# ============================================================

csv_missing_mismatches = 0


for col in EXPECTED_COLUMNS:

    expected_missing = (
        source_df[col]
        .isna()
        .to_numpy()
    )

    exported_missing = (
        roundtrip_df[col]
        .isna()
        .to_numpy()
    )

    csv_missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- CSV MISSING-VALUE FIDELITY ---")

print(
    "Missing-position mismatches:",
    csv_missing_mismatches
)


if csv_missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed "
        "during CSV export."
    )


print(
    "PASS: Missing positions preserved exactly."
)


# ============================================================
# 31. READ FINAL CSV AS RAW TEXT
# ============================================================

#
# Raw-text re-read allows literal formatting checks.
#

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


if raw_csv_df.shape != (
    8897,
    76
):

    raise ValueError(
        "Raw-text CSV structure incorrect."
    )


# ============================================================
# 32. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )

    tiny_csv_count += int(
        values.eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY-VALUE CHECK ---")

print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Unexpected tiny XPORT artifacts found "
        "in final CSV."
    )


print(
    "PASS: No tiny XPORT artifacts in final CSV."
)


# ============================================================
# 33. RAW CSV .0 CHECK
# ============================================================

#
# MCQ_J has been proven to contain NO genuine decimal
# variables.
#
# Therefore it is safe to check ALL 76 columns for
# unnecessary trailing ".0".
#
# Examples:
#
#     SEQN
#         93703.0 -> 93703
#
#     MCQ010
#         1.0 -> 1
#
#     MCD180A
#         50.0 -> 50
#
# IMPORTANT:
#
# Do NOT copy this all-column .0 rule into a mixed
# NHANES file with genuine decimal variables.
#

print("\n--- RAW CSV .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]

    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )

    count = int(
        bad_mask.sum()
    )

    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected '.0' formatting found:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary '.0' in "
    "any of the 76 MCQ_J fields."
)


# ============================================================
# 34. FINAL CSV ZERO CHECK
# ============================================================

csv_zero_counts = {}


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )

    zero_count = int(
        values.eq(0)
        .sum()
    )

    if zero_count > 0:

        csv_zero_counts[col] = zero_count


print("\n--- FINAL CSV ZERO PATTERN ---")

print(
    csv_zero_counts
)


if csv_zero_counts != {}:

    raise ValueError(
        "Unexpected numeric zero values found "
        "in final CSV:\n"
        + str(csv_zero_counts)
    )


print(
    "PASS: Final CSV contains no numeric zeros."
)


# ============================================================
# 35. SCIENTIFIC NOTATION CHECK
# ============================================================

#
# Questionnaire codes and IDs should not be emitted
# using scientific notation.
#

scientific_count = 0


for col in EXPECTED_COLUMNS:

    tokens = raw_csv_df[col]

    scientific_count += int(
        tokens.str.contains(
            r"[eE][+-]\d+",
            regex=True
        )
        .sum()
    )


print("\n--- SCIENTIFIC NOTATION CHECK ---")

print(
    "Scientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 36. FINAL SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if final_seqn.min() != 93703:

    raise ValueError(
        "Final minimum SEQN incorrect."
    )


if final_seqn.max() != 102956:

    raise ValueError(
        "Final maximum SEQN incorrect."
    )


if int(
    final_seqn.isna()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains missing SEQN."
    )


if int(
    final_seqn.duplicated()
    .sum()
) != 0:

    raise ValueError(
        "Final CSV contains duplicate SEQN."
    )


if (
    raw_csv_df["SEQN"]
    .str.endswith(".0")
    .any()
):

    raise ValueError(
        "Final CSV SEQN still contains '.0'."
    )


print(
    "PASS: Final CSV SEQN 93703-102956 valid "
    "and contains no '.0'."
)


# ============================================================
# 37. FINAL CDC FREQUENCY RECHECK FROM CSV
# ============================================================

#
# Recheck the same CDC categorical distributions
# against the final exported CSV rather than df.
#

print(
    "\n--- FINAL CSV CDC FREQUENCY RECHECK ---"
)


for column, (
    expected_values,
    expected_missing
) in CDC_EXACT_CHECKS.items():

    final_values = pd.to_numeric(
        raw_csv_df[column]
        .where(
            raw_csv_df[column] != "",
            np.nan
        ),
        errors="coerce"
    )

    actual_values = (
        final_values
        .dropna()
        .astype(int)
        .value_counts()
        .sort_index()
        .to_dict()
    )

    actual_missing = int(
        final_values
        .isna()
        .sum()
    )

    if actual_values != expected_values:

        raise ValueError(
            f"Final CSV {column} frequencies "
            "differ from CDC."
        )

    if actual_missing != expected_missing:

        raise ValueError(
            f"Final CSV {column} missing count "
            "differs from CDC."
        )


print(
    "PASS: Final CSV CDC frequency checks passed."
)


# ============================================================
# 38. FINAL COMPLETE SOURCE -> CSV COMPARISON
# ============================================================

#
# One last all-variable comparison.
#
# This is deliberately redundant because fidelity
# is the primary objective.
#

final_fidelity_failures = []


for col in EXPECTED_COLUMNS:

    original_values = (
        source_df[col]
        .to_numpy(
            dtype=float
        )
    )

    final_values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    ).to_numpy(
        dtype=float
    )

    if not np.allclose(
        original_values,
        final_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        final_fidelity_failures.append(
            col
        )


if final_fidelity_failures:

    raise ValueError(
        "Final source -> CSV fidelity failures:\n"
        + str(final_fidelity_failures)
    )


print(
    "PASS: Final CSV preserves exact original "
    "numeric values for all 76 variables."
)


# ============================================================
# 39. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 90
)

print(
    "MCQ_J CONVERSION COMPLETE"
)

print(
    "=" * 90
)

print(
    "Cycle: 2017-2018"
)

print(
    "Component: Questionnaire - Medical Conditions"
)

print(
    "Rows:",
    f"{len(df):,}"
)

print(
    "Columns:",
    len(df.columns)
)

print(
    "SEQN range:",
    f"{df['SEQN'].min()}-{df['SEQN'].max()}"
)

print(
    "Original tiny-value artifacts:",
    tiny_total
)

print(
    "Original ordinary numeric zeros:",
    true_zero_total
)

print(
    "Numeric zero restorations:",
    0
)

print(
    "Other numeric corrections:",
    0
)

print(
    "Genuine decimal variables: NONE"
)

print(
    "Whole-number/code variables:",
    len(INTEGER_COLUMNS)
)

print(
    "CDC exact-frequency variables checked:",
    len(CDC_EXACT_CHECKS)
)

print(
    "XPT -> CSV unexpected numeric differences: "
    "0 expected"
)

print(
    "Missing-position differences: "
    "0 expected"
)

print(
    "Tiny values remaining: "
    "0 expected"
)

print(
    "Unnecessary '.0': "
    "0 expected"
)

print(
    "Scientific notation: "
    "0 expected"
)

print(
    "=" * 90
)

NHANES 2017-2018 MCQ_J XPT -> CSV
Rows: 8,897
Columns: 76
PASS: Structure = 8,897 rows x 76 columns.

--- FIELD TYPE PLAN ---
Genuine decimal variables: []
Provisional whole-number/code fields: 76

--- ORIGINAL XPT TINY-VALUE CHECK ---
Total tiny XPORT artifacts: 0
PASS: MCQ_J contains no tiny XPORT artifacts.

--- ORIGINAL NUMERIC ZERO CHECK ---
Ordinary numeric zeros: 0
PASS: Original MCQ_J contains no numeric zeros.

--- FRACTIONAL VALUE CHECK ---
SEQN: 0 fractional observations
MCQ010: 0 fractional observations
MCQ025: 0 fractional observations
MCQ035: 0 fractional observations
MCQ040: 0 fractional observations
MCQ050: 0 fractional observations
AGQ030: 0 fractional observations
MCQ053: 0 fractional observations
MCQ080: 0 fractional observations
MCQ092: 0 fractional observations
MCD093: 0 fractional observations
MCQ149: 0 fractional observations
MCQ151: 0 fractional observations
RHD018: 0 fractional observations
MCQ160A: 0 fractional observations
MCD180A: 0 fractional observations
M